# Synthetic Time Series Data Generation

Generates two tables for the MMF pipeline:

- **`train`** — historical time series with `unique_id`, `ds`, `y`, static features, and dynamic future covariates. Fed to `run_forecast(train_data=...)`.
- **`score`** — future dates with `unique_id`, `ds`, and dynamic future covariates only (no `y`, no static features). Fed to `run_forecast(scoring_data=...)`.

Covariates are generated continuously across the train/score boundary so values are consistent.

## How the data is built

**Target variable (`y`)** — base level + linear trend + seasonal sine wave + noise, then modified by signal covariates below.

**Static features** (constant per series, included in `train` only):
| Column | Values | Role |
|---|---|---|
| `static_category` | A / B / C | **Signal** — shifts base level (+50 / 0 / −30) |
| `static_region` | north / south / east / west | **Signal** — scales seasonality (north 1.5×, south 0.5×, etc.) |
| `static_color` | red / blue / green | **Noise** — no effect on `y` |

**Dynamic future numerical** (vary per series × month, included in both `train` and `score`):
| Column | How generated | Role |
|---|---|---|
| `future_planned_price` | Random walk with occasional jumps | **Signal** — higher price suppresses `y` (elasticity −0.3) |
| `future_temperature` | Seasonal sine + noise, shifted by region | **Signal** — adds to `y` proportionally |
| `future_marketing_spend` | Uniform random [0, 100] | **Noise** — no effect on `y` |

**Dynamic future categorical** (vary per series × month, included in both `train` and `score`):
| Column | Values | Role |
|---|---|---|
| `future_is_promotion` | 0 / 1 (~20% months) | **Signal** — promo months get 1.15× multiplier |
| `future_event_type` | none / minor / major | **Signal** — minor +5, major +20 to `y` |
| `future_channel` | online / instore / wholesale | **Noise** — no effect on `y` |

~10% of training series are then contaminated with spikes, drops, nulls, or gaps.
~0.1% of covariate cells in the scoring table are set to null to exercise downstream null-handling.

## 1. Parameters

In [ ]:
try:
    catalog = dbutils.widgets.get("catalog")
except Exception:
    catalog = "mmf"

try:
    schema = dbutils.widgets.get("schema")
except Exception:
    schema = "synthetic"

try:
    prediction_length = int(dbutils.widgets.get("prediction_length"))
except Exception:
    prediction_length = 12

train_table = f"{catalog}.{schema}.train"
score_table = f"{catalog}.{schema}.score"
print(f"Train table: {train_table}")
print(f"Score table: {score_table}")
print(f"Prediction length: {prediction_length} months")

## 2. Setup — Create Catalog and Schema

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

## 3. Generate Synthetic Time Series

In [ ]:
import numpy as np
import pandas as pd
from datetime import date

np.random.seed(42)

N_SERIES = 1000
N_YEARS = 5
N_TRAIN = N_YEARS * 12
N_TOTAL = N_TRAIN + prediction_length
TRAIN_END = date.today().replace(day=1)

train_dates = pd.date_range(end=TRAIN_END, periods=N_TRAIN, freq="MS")
score_dates = pd.date_range(start=train_dates[-1] + pd.offsets.MonthBegin(1), periods=prediction_length, freq="MS")
all_dates = train_dates.union(score_dates)
t = np.arange(N_TOTAL)

categories = np.random.choice(["A", "B", "C"], size=N_SERIES)
regions = np.random.choice(["north", "south", "east", "west"], size=N_SERIES)
colors = np.random.choice(["red", "blue", "green"], size=N_SERIES)

CATEGORY_OFFSET = {"A": 50.0, "B": 0.0, "C": -30.0}
REGION_SEASON_SCALE = {"north": 1.5, "south": 0.5, "east": 1.0, "west": 1.2}
PRICE_ELASTICITY = -0.3
EVENT_OFFSET = {"none": 0.0, "minor": 5.0, "major": 20.0}
PROMO_MULTIPLIER = 1.15

train_records = []
score_records = []

for i in range(N_SERIES):
    uid = f"series_{i:04d}"
    cat, reg, col = categories[i], regions[i], colors[i]

    trend_slope = np.random.uniform(0.5, 5.0)
    level = np.random.uniform(50, 500)
    seasonality_amp = np.random.uniform(5, 50)
    noise_std = np.random.uniform(1, 20)

    # --- Dynamic future numerical (generated across full date range) ---
    price_steps = np.random.normal(0, 0.5, N_TOTAL)
    price_steps[np.random.choice(N_TOTAL, size=3, replace=False)] += np.random.choice([-5, 5], size=3)
    planned_price = np.cumsum(price_steps) + np.random.uniform(20, 80)
    planned_price = np.clip(planned_price, 5, 150)

    region_phase = {"north": 0, "south": np.pi, "east": np.pi / 2, "west": -np.pi / 2}[reg]
    temperature = 15 + 12 * np.sin(2 * np.pi * t / 12 + region_phase) + np.random.normal(0, 2, N_TOTAL)

    marketing_spend = np.random.uniform(0, 100, N_TOTAL)

    # --- Dynamic future categorical (generated across full date range) ---
    is_promotion = (np.random.rand(N_TOTAL) < 0.20).astype(int)
    event_probs = np.random.rand(N_TOTAL)
    event_type = np.where(event_probs < 0.05, "major", np.where(event_probs < 0.20, "minor", "none"))
    channel = np.random.choice(["online", "instore", "wholesale"], size=N_TOTAL)

    # --- Build y for training period only ---
    seasonal_raw = seasonality_amp * np.sin(2 * np.pi * t[:N_TRAIN] / 12)
    seasonal = REGION_SEASON_SCALE[reg] * seasonal_raw
    noise = np.random.normal(0, noise_std, N_TRAIN)

    y = (
        level
        + CATEGORY_OFFSET[cat]
        + trend_slope * t[:N_TRAIN]
        + seasonal
        + PRICE_ELASTICITY * planned_price[:N_TRAIN]
        + 0.3 * temperature[:N_TRAIN]
        + np.array([EVENT_OFFSET[e] for e in event_type[:N_TRAIN]])
        + noise
    )
    y = np.where(is_promotion[:N_TRAIN] == 1, y * PROMO_MULTIPLIER, y)
    y = np.clip(y, 0, None)

    for j in range(N_TRAIN):
        train_records.append((
            uid, all_dates[j].date(), float(y[j]),
            cat, reg, col,
            float(planned_price[j]), float(temperature[j]), float(marketing_spend[j]),
            int(is_promotion[j]), event_type[j], channel[j],
        ))

    for j in range(N_TRAIN, N_TOTAL):
        score_records.append((
            uid, all_dates[j].date(),
            float(planned_price[j]), float(temperature[j]), float(marketing_spend[j]),
            int(is_promotion[j]), event_type[j], channel[j],
        ))

train_pdf = pd.DataFrame(train_records, columns=[
    "unique_id", "ds", "y",
    "static_category", "static_region", "static_color",
    "future_planned_price", "future_temperature", "future_marketing_spend",
    "future_is_promotion", "future_event_type", "future_channel",
])

score_pdf = pd.DataFrame(score_records, columns=[
    "unique_id", "ds",
    "future_planned_price", "future_temperature", "future_marketing_spend",
    "future_is_promotion", "future_event_type", "future_channel",
])

print(f"=== Train ===")
print(f"  Rows: {len(train_pdf):,}  |  Series: {train_pdf['unique_id'].nunique()}")
print(f"  Date range: {train_pdf['ds'].min()} → {train_pdf['ds'].max()}")
print(f"  Columns: {list(train_pdf.columns)}")
print(f"\n=== Score ===")
print(f"  Rows: {len(score_pdf):,}  |  Series: {score_pdf['unique_id'].nunique()}")
print(f"  Date range: {score_pdf['ds'].min()} → {score_pdf['ds'].max()}")
print(f"  Columns: {list(score_pdf.columns)}")
train_pdf.head()

## 3b. Inject Anomalies and Missing Entries

Contaminates ~10% of series with realistic data-quality issues:
- **Spikes** — sudden values 5–10× the local mean
- **Drops** — values forced to zero or near-zero
- **Nulls** — `y` set to `NaN` (sensor dropout / ETL failure)
- **Gaps** — entire rows removed (missing months)

In [ ]:
rng = np.random.RandomState(99)

all_series = train_pdf["unique_id"].unique()
n_affected = int(len(all_series) * 0.10)
affected_series = rng.choice(all_series, size=n_affected, replace=False)

rng.shuffle(affected_series)
spike_ids = affected_series[: n_affected // 4]
drop_ids  = affected_series[n_affected // 4 : n_affected // 2]
null_ids  = affected_series[n_affected // 2 : 3 * n_affected // 4]
gap_ids   = affected_series[3 * n_affected // 4 :]

for sid in spike_ids:
    mask = train_pdf["unique_id"] == sid
    idx = train_pdf.index[mask]
    n_spikes = rng.randint(2, 5)
    spike_idx = rng.choice(idx, size=min(n_spikes, len(idx)), replace=False)
    train_pdf.loc[spike_idx, "y"] *= rng.uniform(5, 10, size=len(spike_idx))

for sid in drop_ids:
    mask = train_pdf["unique_id"] == sid
    idx = train_pdf.index[mask]
    n_drops = rng.randint(2, 5)
    drop_idx = rng.choice(idx, size=min(n_drops, len(idx)), replace=False)
    train_pdf.loc[drop_idx, "y"] = rng.uniform(0, 1, size=len(drop_idx))

for sid in null_ids:
    mask = train_pdf["unique_id"] == sid
    idx = train_pdf.index[mask]
    n_nulls = rng.randint(3, 7)
    null_idx = rng.choice(idx, size=min(n_nulls, len(idx)), replace=False)
    train_pdf.loc[null_idx, "y"] = np.nan

gap_rows_to_drop = []
for sid in gap_ids:
    mask = train_pdf["unique_id"] == sid
    idx = train_pdf.index[mask].tolist()
    gap_len = rng.randint(3, 7)
    if len(idx) > gap_len + 2:
        start = rng.randint(1, len(idx) - gap_len - 1)
        gap_rows_to_drop.extend(idx[start : start + gap_len])

train_pdf = train_pdf.drop(index=gap_rows_to_drop).reset_index(drop=True)

n_spikes_total = len(spike_ids)
n_drops_total = len(drop_ids)
n_nulls_total = int(train_pdf["y"].isna().sum())
n_gaps_total = len(gap_rows_to_drop)

print(f"Injected anomalies into {n_affected} / {len(all_series)} series:")
print(f"  Spike series : {n_spikes_total}")
print(f"  Drop series  : {n_drops_total}")
print(f"  Null values  : {n_nulls_total}")
print(f"  Removed rows : {n_gaps_total} (gap series: {len(gap_ids)})")
print(f"Final train row count: {len(train_pdf):,}")

## 3c. Inject Sparse Nulls into Scoring Data

Randomly sets ~1% of covariate cells to `NaN`/`None` across the scoring DataFrame.
This exercises the null-handling logic in Skill 1 (Step 6a validation).

In [ ]:
score_rng = np.random.RandomState(77)
NULL_FRACTION = 0.001
covariate_cols = [
    "future_planned_price", "future_temperature", "future_marketing_spend",
    "future_is_promotion", "future_event_type", "future_channel",
]

n_cells = len(score_pdf) * len(covariate_cols)
n_nulls = int(n_cells * NULL_FRACTION)
null_rows = score_rng.randint(0, len(score_pdf), size=n_nulls)
null_cols = score_rng.choice(covariate_cols, size=n_nulls)

for r, c in zip(null_rows, null_cols):
    score_pdf.at[r, c] = None

null_counts = score_pdf[covariate_cols].isna().sum()
print(f"Injected ~{NULL_FRACTION:.0%} nulls into scoring data ({n_nulls} cells across {n_cells:,} total):")
for col, cnt in null_counts.items():
    print(f"  {col}: {cnt} nulls")
print(f"  Total null cells: {int(null_counts.sum())}")

## 4. Save to Delta Tables

In [ ]:
for name, pdf in [(train_table, train_pdf), (score_table, score_pdf)]:
    sdf = spark.createDataFrame(pdf)
    (
        sdf.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(name)
    )
    print(f"Table written: {name}")

## 5. Verify

In [ ]:
print("=== Train table ===")
display(spark.sql(f"""
    SELECT
        COUNT(*)                          AS total_rows,
        COUNT(DISTINCT unique_id)         AS n_series,
        MIN(ds)                           AS min_date,
        MAX(ds)                           AS max_date,
        ROUND(AVG(y), 2)                  AS avg_y,
        SUM(CASE WHEN y IS NULL THEN 1 ELSE 0 END) AS null_count,
        ROUND(MAX(y), 2)                  AS max_y,
        ROUND(MIN(y), 2)                  AS min_y
    FROM {train_table}
""").toPandas())

print("\nTrain covariate distributions:")
display(spark.sql(f"""
    SELECT
        COUNT(DISTINCT static_category) AS n_categories,
        COUNT(DISTINCT static_region)   AS n_regions,
        COUNT(DISTINCT static_color)    AS n_colors,
        ROUND(AVG(future_planned_price), 2)   AS avg_price,
        ROUND(AVG(future_temperature), 2)     AS avg_temp,
        ROUND(AVG(future_marketing_spend), 2) AS avg_mktg,
        ROUND(AVG(future_is_promotion), 3)    AS promo_rate,
        COUNT(DISTINCT future_event_type)     AS n_event_types,
        COUNT(DISTINCT future_channel)        AS n_channels
    FROM {train_table}
""").toPandas())

print("\n=== Score table ===")
display(spark.sql(f"""
    SELECT
        COUNT(*)                  AS total_rows,
        COUNT(DISTINCT unique_id) AS n_series,
        MIN(ds)                   AS min_date,
        MAX(ds)                   AS max_date
    FROM {score_table}
""").toPandas())

print("\nScore columns:")
spark.sql(f"DESCRIBE {score_table}").show(truncate=False)

In [ ]:
print("Train: months per series (expect 60 except gap-affected)")
spark.sql(f"""
    SELECT n_months, COUNT(*) AS n_series
    FROM (SELECT unique_id, COUNT(*) AS n_months FROM {train_table} GROUP BY unique_id)
    GROUP BY n_months ORDER BY n_months
""").show()

print(f"Score: months per series (expect {prediction_length} for all)")
spark.sql(f"""
    SELECT n_months, COUNT(*) AS n_series
    FROM (SELECT unique_id, COUNT(*) AS n_months FROM {score_table} GROUP BY unique_id)
    GROUP BY n_months ORDER BY n_months
""").show()

series_coverage = spark.sql(f"""
    SELECT
        (SELECT COUNT(DISTINCT unique_id) FROM {train_table}) AS train_series,
        (SELECT COUNT(DISTINCT unique_id) FROM {score_table}) AS score_series
""").toPandas()
print("Series coverage (should match):")
display(series_coverage)